# [LAB] LLM API 실습

같은 질문 "LLM이 무엇인지 3문장으로 설명해줘."를 세 가지 방법으로 LLM에 보내 봅니다. 방법이 바뀔 때마다 무엇이 편해지는지 비교하는 것이 이 노트북의 목표입니다.

- **2장 요청을 직접 보내기** (예제): 주소·헤더·요청 본문을 손으로 만들어 보냅니다. `# ✍️ CHANGE_HERE:` 줄의 질문을 바꿔 다시 보내 보고, AI에게 설명을 받습니다.
- **3장 SDK로 묻기** (예제): 도구(SDK)가 요청을 대신 보냅니다. 한 번 묻기, 이어서 묻기, 조각으로 받기를 실행합니다. `# ✍️ CHANGE_HERE:` 줄의 말투 규칙을 바꿔 보고, AI에게 설명을 받습니다.
- **4장 LangChain으로 호출하기** (예제): 같은 일을 `get_llm()` 한 줄로 준비한 모델로 합니다. 시스템 메시지를 바꿔 답의 말투가 달라지는지 봅니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다.

**목차**

- 1. 실습 방법
- 2. 요청을 직접 보내기 (예제)
  - 2.1 요청 본문 만들기 (예제)
  - 2.2 보내고 응답 받기 (예제)
  - 2.3 참고: 다른 요청 방식(Responses API)
  - 2.4 AI에게 물어보기
- 3. SDK로 한 번 묻기, 이어 묻기, 조각으로 받기 (예제)
  - 3.1 한 번 묻기 (예제)
  - 3.2 이어서 묻기 (예제)
  - 3.3 조각으로 받기 (예제)
  - 3.4 참고: temperature와 top_p
  - 3.5 AI에게 물어보기
- 4. LangChain으로 호출하기 (예제)
  - 4.1 역할 메시지로 한 번에 받기 (예제)
  - 4.2 조각으로 받기 (예제)
  - 4.3 AI에게 물어보기
- 5. 정리

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))  # 프로젝트의 .env 파일을 읽습니다

# 값은 출력하지 않습니다. 빠진 설정이 있는지만 봅니다.
필요한_설정 = {
    "LLM_PROVIDER": {
        "openai": ["OPENAI_API_KEY", "OPENAI_MODEL"],
        "samsung": [
            "api_base_url",
            "model",
            "credential_key",
            "send_system_name",
            "user_id",
        ],
    }
}
빠진_이름 = []
for 설정, 종류별 in 필요한_설정.items():
    종류 = os.getenv(설정, "samsung").strip().lower()
    빠진_이름 += [n for n in 종류별.get(종류, [설정]) if not os.getenv(n)]

import config.llm  # 실습 코드가 쓰는 설정 모듈을 불러옵니다

if 빠진_이름:
    print(
        f"준비가 덜 되었습니다. 빠진 설정 {len(빠진_이름)}개가 있습니다. 강사에게 알려 주세요."
    )
    print("(강사 확인용:", ", ".join(빠진_이름), ")")
else:
    print("준비 완료: 실습에 필요한 설정이 모두 있습니다.")

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 강사에게 알립니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 강사에게 알립니다.

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/01-llm-api/01_requests.py`·`02_openai.py`·`03_langchain.py`. 빌드할 때 셀로 옮겨 넣으므로 수강생은 .py 파일을 보지 않습니다.
- 원본에는 환경에 따라 주소·모델·인증을 고르는 분기가 있습니다(`LLM_PROVIDER`). openai 분기는 사외 준비용으로 우리가 더한 것이라 수강생 글에서는 다루지 않습니다. 2장의 보이는 셀(요청 본문)에 `provider`·`gpt-6-luna` 분기가 그대로 보이니, 물으면 "연결 환경에 맞춘 설정"이라고만 답합니다.
- CodeMate가 노트북에서 선택한 셀을 읽는지, 셀을 직접 고칠 수 있는지는 강의장에서 아직 확인하지 않았습니다. 못 읽으면 셀 코드를 지시문 아래에 붙여 넣게 하고, 못 고치면 채팅 답의 코드를 수강생이 빈칸에 붙여 넣게 합니다.
- CodeMate 지시문 완성본은 주지 않습니다(2026-09-28 결정). 힌트와 성공 기준만 주고 수강생이 지시문을 씁니다.
- 성공 기준은 자동 검사하지 않습니다(2026-09-28 결정). 수강생이 실행 결과를 보고 스스로 판단합니다.

</details>

## 1. 실습 방법

실습은 두 종류입니다. 제목 옆 표시와 코드 주석으로 구분합니다.

- **예제** `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- **과제** `# ✍️ WRITE_HERE:` 아래 빈칸(`...`)을 AI와 함께 채웁니다. **프롬프트 힌트**를 보고 지시를 직접 씁니다.

실행한 뒤에는 **성공 기준**과 비교해 스스로 판단합니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. 과제는 그다음 접힌 "정답 보기"와 비교합니다.

## 2. 요청을 직접 보내기 <small>(예제)</small>

`requests`라는 도구로 LLM 서버 주소에 요청을 직접 보냅니다.
요청 본문(payload)은 JSON(이름과 값을 짝지어 적는 데이터 형식)으로 만들고, 응답도 JSON으로 받습니다.

코드는 완성되어 있습니다. 준비 코드부터 차례로 실행하고, 출력된 응답 JSON을 읽어 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 줄의 질문을 바꿔 다시 보내 봅니다. 요청 본문 셀 아래쪽의 `if` 부분은 연결 환경에 맞춰 설정을 조금 바꾸는 코드라 건너뛰어도 됩니다.

**왜 배우나:** 3장의 SDK와 4장의 LangChain도 속에서는 이런 요청을 보냅니다. 날것의 요청을 한 번 보아 두면, API 오류가 났을 때 어디부터 살펴야 할지 알 수 있습니다.

준비 코드입니다. 필요한 도구를 불러오고, 요청을 보낼 주소·모델 이름·인증 정보를 준비합니다. 접혀 있으니 펼치지 않아도 됩니다. 실행만 합니다.

In [ ]:
import os
from pprint import pprint

import requests
from dotenv import load_dotenv

load_dotenv()

provider = os.getenv("LLM_PROVIDER", "samsung").strip().lower()
if provider == "openai":
    api_base_url = "https://api.openai.com/v1/chat/completions"
    model = os.getenv("OPENAI_MODEL")
    api_key = os.getenv("OPENAI_API_KEY")
    if not model or not api_key:
        raise RuntimeError("OPENAI_MODEL과 OPENAI_API_KEY가 필요합니다.")
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
elif provider == "samsung":
    # 사내 게이트웨이 주소·모델·인증 헤더는 config/samsung.py에 모아 두었다.
    from config.samsung import llm_config

    samsung = llm_config()
    api_base_url = f"{samsung['base_url'].rstrip('/')}/chat/completions"
    model = samsung["model"]
    headers = {**samsung["headers"], "Content-Type": "application/json"}
else:
    raise ValueError(f"지원하지 않는 LLM_PROVIDER: {provider}")

<details><summary>강사용 메모</summary>

원본: `src/raw/Chapter01/01-llm-api/01_requests.py`. 보이는 셀은 원본 30~54줄, 참고 코드는 57~83줄입니다.

</details>

### 2.1 요청 본문 만들기 <small>(예제)</small>

**✍️ CHANGE_HERE: 질문을 바꿔 봅니다**

먼저 그대로 실행해 답을 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 줄의 따옴표 안 질문을 원하는 질문으로 바꾸고, 이 셀과 다음 셀을 다시 실행합니다. 따옴표와 쉼표는 지우지 않습니다.

In [ ]:
payload = {
    "model": model,
    "messages": [
        # ✍️ CHANGE_HERE: 질문을 바꿔 봅니다
        {"role": "user", "content": "LLM이 무엇인지 3문장으로 설명해줘."},
    ],
    "temperature": 0.5,
    "stream": False,
}
if provider == "samsung":
    payload["include_reasoning"] = False
elif model == "gpt-6-luna":
    # GPT-6 Luna의 Chat Completions는 기본 추론 설정에서 temperature를 받지 않는다.
    payload.pop("temperature")
    payload["reasoning_effort"] = "none"

### 2.2 보내고 응답 받기 <small>(예제)</small>

In [ ]:
new_response = requests.post(api_base_url, headers=headers, json=payload, timeout=30)
new_response.raise_for_status()
recv_json = new_response.json()

print("### 응답 Json")
pprint(recv_json)
print("### 결과 메시지")
print(recv_json["choices"][0]["message"]["content"])

**성공 기준**

- 🟢 바꾼 질문으로 다시 보냈고, 답도 바뀐 질문에 맞는다
- 🟢 오류 없이 "### 응답 Json"과 "### 결과 메시지"가 모두 출력되었다

### 2.3 참고: 다른 요청 방식(Responses API)

문자열로 감싸 두어 실행되지 않는 참고 코드입니다. 읽어만 봅니다.

```python
# OpenAI Responses API 엔드포인트
# - 기존 Chat Completions(/v1/chat/completions)의 후속 API
# - input 필드 하나로 단순 문자열과 대화 내역 모두 처리 가능
api_key = os.getenv('OPENAI_API_KEY')
model = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')

if not api_key:
    raise RuntimeError('OPENAI_API_KEY 환경변수를 설정해주세요.')

url = 'https://api.openai.com/v1/responses'
headers = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type': 'application/json',
}

payload = {
    'model': model,
    'input': 'LLM이 무엇인지 3문장으로 설명해줘.',
}

response = requests.post(url, headers=headers, json=payload, timeout=30)
response.raise_for_status()

data = response.json()
pprint(data)
```

**볼 지점**
- "### 결과 메시지" 아래 줄이 모델의 답변 텍스트입니다.
- "### 응답 Json" 아래에서 `choices` → `message` → `content`를 따라가면 같은 텍스트가 있습니다. 코드는 이 경로로 답을 꺼냈습니다.
- `usage`를 찾아봅니다. `prompt_tokens`는 입력 토큰 수, `completion_tokens`는 출력 토큰 수, `total_tokens`는 둘의 합입니다.
- 숫자는 실행할 때마다 다릅니다. `usage`가 보이지 않으면 "이 환경에서는 제공되지 않음"이라고 적어 둡니다.

<details><summary>강사용 메모</summary>

- 원본 9~28줄의 분기에서 `.env`의 `LLM_PROVIDER` 값에 따라 주소·모델·헤더가 정해집니다. 기본값은 samsung입니다.
- 원본 38~43줄(요청 본문 셀 아래쪽 if): samsung이면 `include_reasoning=False`를 더하고, openai이면서 모델이 `gpt-6-luna`이면 temperature를 빼고 `reasoning_effort='none'`을 넣습니다.
- 사내 게이트웨이 응답에 `usage`가 들어오는지는 아직 확인하지 않았습니다. 토큰 수는 환경마다 다릅니다.

</details>

### 2.4 AI에게 물어보기

**프롬프트 힌트** 요청 본문과 보내고 응답 받기 셀을 붙여 넣고, 코드는 고치지 말고 줄마다 한 문장으로 설명해 달라고 합니다. 응답 JSON의 `usage`에 어떤 값이 나올지도 예측해 달라고 합니다.

**성공 기준**

- 🟢 AI가 예측한 `usage`의 이름이 실행 결과의 `usage`와 맞는다
- 🟢 답변 텍스트가 든 경로(`choices` → `message` → `content`)를 설명에서 짚었다
- ⛔ 코드를 고치거나 새로 써 왔다 → 고치지 말라고 다시 적습니다
- ⛔ 설명이 실행 결과와 다르다 → 그 코드 줄을 들어 다시 묻습니다

## 3. SDK로 한 번 묻기, 이어 묻기, 조각으로 받기 <small>(예제)</small>

이번에는 OpenAI SDK(개발 도구 묶음)로 같은 일을 합니다. 2장에서 손으로 만든 주소·헤더·JSON 처리를 `client`(요청을 대신 보내 주는 객체)가 맡습니다. 요청 방식은 `messages` 대신 `input`에 내용을 담는 Responses API(새 요청 방식)입니다.

한 번 묻기(Single Turn), 이어서 묻기(Multi Turn), 조각으로 받기(Streaming)를 차례로 실행합니다. 이어서 묻기에서는 `# ✍️ CHANGE_HERE:` 줄의 말투 규칙을 바꿔 봅니다. 요청을 네 번 보내므로 답이 모두 나오기까지 조금 걸립니다.

**왜 배우나:** 모델은 앞 대화를 기억하지 않습니다. "이어서 묻기" 셀에서 코드가 이전 대화를 매번 다시 넣어 보내는 모습을 직접 봅니다. 메모리나 Checkpointer 같은 기능은 이 "다시 넣기"를 도구가 대신 해 주는 방법입니다.

이 장의 준비 코드입니다. 요청을 대신 보내 줄 `client`와 모델 이름을 만듭니다. 내용은 몰라도 되니 실행만 하고 넘어갑니다.

In [ ]:
import os
from collections import Counter
from pprint import pprint

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

provider = os.getenv("LLM_PROVIDER", "samsung").strip().lower()
if provider == "openai":
    model = os.getenv("OPENAI_MODEL")
    api_key = os.getenv("OPENAI_API_KEY")
    if not model or not api_key:
        raise RuntimeError("OPENAI_MODEL과 OPENAI_API_KEY가 필요합니다.")
    client = OpenAI(api_key=api_key)
elif provider == "samsung":
    # 사내 게이트웨이 주소·모델·인증 헤더는 config/samsung.py에 모아 두었다.
    from config.samsung import llm_config

    samsung = llm_config()
    model = samsung["model"]
    client = OpenAI(
        api_key="samsung-gateway",  # 실제 사내 인증은 x-dep-ticket 헤더에서 처리
        base_url=samsung["base_url"],
        default_headers=samsung["headers"],
    )
else:
    raise ValueError(f"지원하지 않는 LLM_PROVIDER: {provider}")


# OpenAI SDK 예제는 사외에서는 공식 API, 사내에서는 호환 게이트웨이를 사용한다.

<details><summary>강사용 메모</summary>

원본: `src/raw/Chapter01/01-llm-api/02_openai.py`. 한 번 묻기 34~42줄, 이어서 묻기 44~58줄, 조각으로 받기 61~74줄, 참고 77~89줄.

</details>

### 3.1 한 번 묻기 <small>(예제)</small>

In [ ]:
# ── Single Turn ───────────────────────────────────────────────────────────────
# input에 문자열을 직접 전달하면 사용자 메시지로 처리
response = client.responses.create(
    model=model,
    input="LLM이 무엇인지 3문장으로 설명해줘.",
)
print(response.output_text)
print()

### 3.2 이어서 묻기 <small>(예제)</small>

**✍️ CHANGE_HERE: 말투 규칙을 바꿔 봅니다**

먼저 그대로 실행해 두 답을 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 줄의 `developer` 규칙을 바꿔 다시 실행합니다(예: "반말로, 한 문장으로 답해줘."). 두 번째 답에도 바꾼 규칙이 지켜지는지 봅니다.

In [ ]:
# ── Multi Turn ────────────────────────────────────────────────────────────────
# input에 역할(role)이 있는 메시지 리스트를 전달하면 대화 맥락 유지
# developer: 시스템 지시 / user: 사용자 입력 / assistant: AI 응답
context = [
    {
        "role": "developer",
        # ✍️ CHANGE_HERE: 말투 규칙을 바꿔 봅니다
        "content": "너는 단답형 답변 챗봇이야. 경어체를 유지하면서, 가능하면 단답형으로 답변해줘.",
    },
    {"role": "user", "content": "대한민국의 수도는 어디지?"},
]
response_1 = client.responses.create(model=model, input=context)
print(response_1.output_text)

# 이전 응답(response_1.output)을 context에 추가해 대화를 이어감
context += response_1.output
context += [{"role": "user", "content": "그곳의 인구는 어느정도야?"}]
response_2 = client.responses.create(model=model, input=context)
print(response_2.output_text)

**성공 기준**

- 🟢 바꾼 말투 규칙이 두 답 모두에 지켜졌다
- 🟢 두 번째 답이 "그곳"을 서울로 알아듣고 답했다

### 3.3 조각으로 받기 <small>(예제)</small>

In [ ]:
# ── Streaming ─────────────────────────────────────────────────────────────────
# stream()을 사용하면 LLM이 생성하는 토큰을 실시간으로 받을 수 있음
# output_text.delta 이벤트에 실제 텍스트 조각이 담겨 있음
with client.responses.stream(
    model=model,
    input="LLM이 무엇인지 3문장으로 설명해줘.",
) as stream:
    event_types = []
    for event in stream:
        event_types.append(event.type)
        if "output_text.delta" in event.type:
            print(event.delta, end="")
    print()
    pprint(Counter(event_types))

### 3.4 참고: temperature와 top_p

문자열로 감싸 두어 실행되지 않는 참고 코드입니다. 읽어만 봅니다.

```python
# temperature: 0에 가까울수록 일관된 답변, 높을수록 창의적(최대 2.0)
# top_p: 확률 상위 p% 토큰만 사용. temperature와 동시 조정은 권장하지 않음
# 주의: gpt-5 계열은 temperature 미지원
response = client.responses.create(
    model=model,
    input='LLM이 무엇인지 3문장으로 설명해줘.',
    temperature=0.2,
    top_p=0.1,
)
print(response.output_text)
```

**볼 지점**
- 첫 답은 2장과 같은 질문에 대한 답입니다. 2장의 답과 문장이 똑같은지 비교해 봅니다.
- "대한민국의 수도는 어디지?"에 짧은 존댓말로 답했는지 봅니다. 이 규칙은 `developer` 메시지에 들어 있습니다.
- "그곳의 인구는 어느정도야?"에 서울의 인구로 답했는지 봅니다. 앞 대화를 `context`에 붙여 다시 보냈기 때문입니다.
- 마지막 답은 조각으로 이어서 출력됩니다. 맨 아래 `Counter(...)`에서 `response.output_text.delta` 옆의 수가 텍스트 조각 이벤트가 온 횟수입니다.

<details><summary>강사용 메모</summary>

- 사내 게이트웨이가 Responses API(`client.responses`)를 지원하는지는 아직 확인하지 않았습니다. 사내에서 오류가 나면 CodeMate 설명만 하고 넘어갑니다.
- 사내 경로에서는 인증을 헤더로 하므로 `api_key` 자리에 이름만 채웁니다(원본 23줄, 접힌 준비 코드 안).

</details>

### 3.5 AI에게 물어보기

**프롬프트 힌트** 이어서 묻기 셀을 붙여 넣고, 두 요청에 각각 무엇이 담겨 가는지 순서대로 적어 달라고 합니다. "모델이 기억해서"인지 "코드가 다시 보내서"인지 구분해 달라고 덧붙입니다.

**성공 기준**

- 🟢 두 번째 요청의 내용에 첫 질문, 첫 답, 새 질문이 모두 들어 있다
- 🟢 "그곳"을 알아듣게 하는 코드 줄(`context += ...`)을 짚었다
- ⛔ "모델이 기억해서"라고만 설명했다 → 어느 줄이 대화를 다시 넣는지 다시 묻습니다
- ⛔ 실행해 보지 않은 예측을 사실처럼 말했다 → 이 노트북에서 확인한 것과 추측을 나눠 달라고 합니다

## 4. LangChain으로 호출하기 <small>(예제)</small>

같은 질문을 LangChain(LLM 호출을 묶어 쓰게 해 주는 도구)으로 보냅니다. 모델 준비는 접힌 준비 코드 안의 `llm = get_llm()` 한 줄로 끝납니다. 2·3장에서 길게 쓰던 준비 코드가 이 함수 하나에 모여 있습니다.

완성된 코드를 실행해 봅니다. 3장에서 한 일과 짝이 맞습니다. 역할을 나눈 메시지는 `SystemMessage`·`HumanMessage`, 한 번에 받기는 `invoke()`, 조각으로 받기는 `stream()`입니다. LangChain은 다음 시간에 자세히 배웁니다.

**왜 배우나:** 앞으로 만드는 에이전트도 모두 이 `get_llm()`과 메시지 목록으로 모델을 호출합니다.

준비 코드는 메시지 종류(`SystemMessage`, `HumanMessage`)를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 펼쳐서 `get_llm()` 줄만 확인해 봐도 좋습니다. 실행을 잊지 않습니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage

from config.llm import get_llm

load_dotenv()


llm = get_llm()

<details><summary>강사용 메모</summary>

원본: `src/solution/Chapter01/01-llm-api/03_langchain.py`(가독성 우선으로 솔루션을 씀). 받은 원본 `src/raw/…/03_langchain.py`는 TODO 3곳에 정답이 채워져 있고, 스트리밍이 `print(chunk)`, 8~11줄에 정리되지 않은 메모가 있습니다. `get_llm()`은 `src/config/llm.py`에 있습니다.
- 2026-09-28 결정: LangChain은 1-3에서 배우므로 1-2에서는 과제가 아니라 예제로 실행만 합니다(커리큘럼 1.2 "get_llm() 래퍼를 통한 모델 호출 일원화").

</details>

### 4.1 역할 메시지로 한 번에 받기 <small>(예제)</small>

**✍️ CHANGE_HERE: 역할을 바꿔 봅니다**

먼저 그대로 실행해 답을 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 역할을 바꿔 다시 실행합니다(예: "너는 초등학생에게 설명하는 선생님이야.").

In [ ]:
# ── 기본 invoke ───────────────────────────────────────────────────────────────
# SystemMessage: LLM의 역할/행동 지침 설정
# HumanMessage: 사용자 입력
messages = [
    # ✍️ CHANGE_HERE: 역할을 바꿔 봅니다
    SystemMessage(content="너는 초보자를 위한 AI 튜터야."),
    HumanMessage(content="LLM이 무엇인지 3문장으로 설명해줘."),
]

response = llm.invoke(messages)
print(response.content)
print()

**성공 기준**

- 🟢 바꾼 역할에 맞게 답의 말투나 수준이 달라졌다
- 🟢 답변 텍스트만 한 번 출력되었다

### 4.2 조각으로 받기 <small>(예제)</small>

In [ ]:
# ── Streaming ─────────────────────────────────────────────────────────────────
# stream()은 토큰 단위로 응답을 yield함
# end='' 로 줄바꿈 없이 실시간 출력
for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)
print()

**성공 기준**

- 🟢 같은 답이 타자 치듯 한 문단으로 이어져 출력되었다

**볼 지점**
- 두 셀 모두 같은 `messages`를 보냅니다. 첫 셀은 답을 한 번에, 둘째 셀은 조각으로 받습니다.
- 둘째 셀의 `print(chunk.content, end="", flush=True)`가 조각의 텍스트만 줄바꿈 없이 바로 보여 줍니다.

<details><summary>강사용 메모</summary>

- 사내 환경에서 조각에 `usage_metadata`가 오는지는 아직 확인하지 않았습니다.

</details>

### 4.3 AI에게 물어보기

**프롬프트 힌트** 두 셀을 붙여 넣고, 3장의 SDK 코드와 비교해 각 줄이 무엇에 해당하는지 짝지어 설명해 달라고 합니다. 코드는 고치지 말라고 적습니다.

**성공 기준**

- 🟢 `SystemMessage`·`HumanMessage`가 3장의 `developer`·`user` 메시지에 해당한다고 짚었다
- 🟢 `invoke()`와 `stream()`의 차이를 한 번에 받기·조각으로 받기로 설명했다
- ⛔ 코드를 고치거나 새로 써 왔다 → 고치지 말라고 다시 적습니다

## 5. 정리

같은 질문을 세 가지 방법으로 보냈습니다.

| 장 | 보내는 방법 | 답변 텍스트를 꺼내는 곳 |
|---|---|---|
| 2. 요청을 직접 보내기 | 주소·헤더·JSON을 직접 만들어 보냄 | `choices[0].message.content` |
| 3. SDK로 묻기 | SDK의 `client`가 대신 보냄 | `output_text` |
| 4. LangChain으로 호출하기 | `get_llm()`으로 준비한 모델에 메시지 목록을 보냄 | `content` |

방법은 달라도 "메시지와 설정값을 보내고, 응답에서 답변 텍스트를 꺼낸다"는 흐름은 같습니다. 앞 대화를 이어 가려면 코드가 이전 대화를 다시 넣어 보내야 한다는 점도 확인했습니다.

값을 바꿔 다시 실행하고, 결과를 성공 기준(🟢/⛔)과 비교해 스스로 판단했습니다.